# FFM_3_2 — mesh2z：网格点云 → RFF → ridge → 每只耳的 z

**问题**：从一只耳（及头部）的 3-D 网格，能预测这只耳在冻结 FFM26 里的 latent 吗？
预测到什么程度——用 dB 说话。

**设计（已与 Harry 确认，2026-10-01）**
- **按耳预测**：左耳点云 → 左耳 z；右耳点云（已镜像进左耳坐标系）→ 右耳 z。一个模型。
  按人平均会白丢 1.2 dB（FFM_3_1：上限从 1.40 退到 2.60 dB）。
- **交叉验证按人分组**：一个人的两只耳永远在同一折。统计单位是人。
- **特征**：RFF 均值嵌入 $\phi(S)=\frac1P\sum_i[\cos 2\pi B\mathbf p_i,\ \sin 2\pi B\mathbf p_i]$，
  $B\in\mathbb R^{M\times3}$ 对所有点共享，σ 单位 mm⁻¹（1/σ 是几何尺度）。
  对照：9 个「笨」几何量。RFF 必须赢过它们。
- **回归**：ridge（核形式，有解析解，无 epoch）；α 由按人分组的内层 CV 选；目标在每折内做 PCA
  （对 ridge 无损，只是让计算变快）。
- **只用 164 个开发集受试者**。36 个锁定的人在这里**不读取任何数据**。

**事先写下的停止规则**（与 landmark2z 相同）：最好配置的 CV-R²(z) < 0.10 → 不再加容量，
写上界结论。由于最好配置是在 15 个里挑出来的，显著性检验用**最大值的置换零分布**。

**读数**（全部按 pooled RMSE，与 `evaluate()` 同口径）
- 端到端 dB，以及在 ④ 均值 HRTF → ① AE 重建 这段空间里收回的百分比
- 认人：预测出的 HRTF 在 164 个同侧耳里离本人最近的比例（随机 = 0.6%）
- CV-R²(z)：内部信号，不进给导师的表

**输出**：`results.json`、图，写到 `OUT_DIR`。

In [ ]:
# ==========================================================================
# Cell 0 — 环境 · 路径
# ==========================================================================
import os, sys, json, csv, zipfile, hashlib, time, warnings
warnings.filterwarnings('ignore')
import numpy as np

IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE = '/content/drive/MyDrive'
    SAVE_DIR = f'{DRIVE}/HRTF_Checkpoints'
    os.system('pip install healpy -q')
    PATHS = dict(
        latents = f'{SAVE_DIR}/mesh2z_sonicom_latents_v1.npz',
        samples = f'{DRIVE}/ELEC5020/SONICOM/mesh2z_samples',
        ckpt    = f'{SAVE_DIR}/FFM26_best.pth',
        sonicom = f'{DRIVE}/ELEC5020/SONICOM.zip',
    )
    OUT_DIR = f'{SAVE_DIR}/mesh2z_rff_v1'
else:
    H = os.path.expanduser('~/Documents')
    R = f'{H}/26S2/ELEC5021/hrtf-GitHub'
    PATHS = dict(
        latents = f'{R}/mesh2z/output/mesh2z_sonicom_latents_v1.npz',
        samples = f'{H}/26S1/ELEC5020/SONICOM/mesh2z_samples',
        ckpt    = os.environ.get('FFM26_CKPT', f'{R}/results/FFM26_best.pth'),
        sonicom = f'{H}/26S1/ELEC5020/SONICOM',
    )
    OUT_DIR = os.environ.get('MESH2Z_OUT', f'{R}/mesh2z/output/rff_v1')
os.makedirs(OUT_DIR, exist_ok=True)
for k, p in PATHS.items():
    print(f'{k:8s} {"OK " if os.path.exists(p) else "★缺失"} {p}')
assert all(os.path.exists(p) for p in PATHS.values()), '先把缺失的文件放好'

In [ ]:
# ==========================================================================
# Cell 1 — 配置
# ==========================================================================
CFG = dict(
    SIGMAS     = [0.005, 0.01, 0.02, 0.05, 0.1],   # mm⁻¹；1/σ = 200, 100, 50, 20, 10 mm
    M          = 256,          # 每个 σ 的随机频率数 -> 512 维（cos + sin）
    N_EAR_PTS  = 20000,        # 耳廓截取区的点（文件里存了 2 万）
    N_HEAD_PTS = 20000,        # 均值嵌入的蒙特卡洛误差 ~1/√P，2 万点足够
    Z_CUT      = -75.0,        # mm；整头只用 z > Z_CUT。脖子截断高度在 -148..-81 之间，因人而异
    PINNA_DY   = 3.0,          # mm；笨特征里「伸出皮肤的耳廓」= 比耳道更靠外 3 mm 以上
    N_FOLDS    = 10,
    N_INNER    = 5,
    ALPHA_REL  = list(np.logspace(-4, 5, 37)),     # α = c · trace(K)/n；上端要够大，无信号的配置会选到这里
    N_PERM     = 100,          # 最大值零分布；想更细就改大（每次约几秒）
    LC_SIZES   = [41, 82, 123, 164],
    LC_REPEATS = 5,
    STOP_R2    = 0.10,
    DECODE     = True,         # 端到端 dB 需要解码器；本地无 GPU 冒烟测试时可关掉
    E2E_CONFIGS= 'all',        # 'all' 或配置名列表
    SMOKE_N    = None,         # 本地冒烟测试：只用前 N 个开发集受试者
    SEED       = 0,
    NOTCH_HZ   = (6000, 12000),
    DID        = 2,
)
print(json.dumps({k: (v if not isinstance(v, list) or len(v) < 8 else f'{len(v)} values') for k, v in CFG.items()}, indent=1))
RNG = np.random.default_rng(CFG['SEED'])

In [ ]:
# ==========================================================================
# Cell 2 - 模型定义（与 FFM_2_3 / FFM_2_6 逐字相同，构建时复制进来）
# ==========================================================================
import numpy as np
import torch
import torch.nn as nn
import healpy as hp


# ---------------------------------------------- spherical convolution
class OSLOConv(nn.Module):
    """Anisotropic convolution on HEALPix: each pixel plus its 8 neighbours, 9 independent weights."""
    def __init__(self, in_channels, out_channels, nside=16):
        super().__init__()
        self.npix = hp.nside2npix(nside)
        neighbors = hp.get_all_neighbours(nside, np.arange(self.npix))
        rf = np.concatenate([neighbors, np.arange(self.npix)[np.newaxis, :]], axis=0)
        rf[rf == -1] = self.npix
        self.register_buffer('neighborhood', torch.tensor(rf, dtype=torch.int64))
        self.weight = nn.Parameter(torch.empty(out_channels, in_channels, 9))
        self.bias = nn.Parameter(torch.empty(out_channels))
        nn.init.kaiming_uniform_(self.weight, a=np.sqrt(5))
        nn.init.zeros_(self.bias)

    def forward(self, x):
        B, C, _ = x.shape
        pad = torch.zeros(B, C, 1, device=x.device, dtype=x.dtype)
        g = torch.cat([x, pad], dim=2)[:, :, self.neighborhood]
        return torch.einsum('bcqn,ocq->bon', g, self.weight) + self.bias.view(1, -1, 1)


class OSLOResBlock(nn.Module):
    """Frequency-isolated residual block, operating on (B*F, C, N).
    * v2: ReLU -> GELU. HRTF log-magnitude is bipolar (negative means attenuation), so
      ReLU truncates the negative half; GELU is smooth, keeps negatives, and matches the fc layers."""
    def __init__(self, in_channels, out_channels, nside=16, num_groups=8):
        super().__init__()
        self.shortcut = (nn.Conv1d(in_channels, out_channels, 1)
                         if in_channels != out_channels else nn.Identity())
        self.conv1 = OSLOConv(in_channels, out_channels, nside)
        self.gn1 = nn.GroupNorm(num_groups, out_channels)
        self.conv2 = OSLOConv(out_channels, out_channels, nside)
        self.gn2 = nn.GroupNorm(num_groups, out_channels)
        self.act = nn.GELU()

    def forward(self, x):
        res = self.shortcut(x)
        out = self.act(self.gn1(self.conv1(x)))
        out = self.gn2(self.conv2(out))
        return self.act(out + res)


# ---------------------------------------------- per-dataset FiLM
class DatasetFiLM(nn.Module):
    """Learn one PER-FREQUENCY scale/shift curve for each dataset.

    ** Constraint 1: frequency only, NEVER direction.
      A per-frequency affine map is one EQ curve (93 dof per dataset). All it can absorb is
      "colouration common to the whole dataset and independent of direction" — loudspeaker and
      microphone response, equalisation convention, overall level. By construction it CANNOT
      express a direction-dependent difference, so population anatomy (ear-size distributions
      differ across databases) must still live in z. Contrast: subtracting a per-database mean
      over (frequency x direction) is 3072x93 dof, enough to remove the real population structure
      as well — exactly the wrong direction Craig pointed at. A factor of 3072 in expressive power.

    ** Constraint 2: centre across datasets.
      Without centring, log_gamma/beta learn a global EQ that cancels against the network's own
      output gain — a redundant degree of freedom. After the cross-database mean is removed FiLM
      expresses only RELATIVE differences; the absolute part stays in the network. Exactly the identity when num_datasets=1.
    """
    def __init__(self, num_datasets, num_bins):
        super().__init__()
        self.num_datasets = num_datasets
        self.log_gamma = nn.Parameter(torch.zeros(num_datasets, num_bins))
        self.beta = nn.Parameter(torch.zeros(num_datasets, num_bins))

    def params(self, dataset_id):
        lg = self.log_gamma - self.log_gamma.mean(0, keepdim=True)
        bt = self.beta - self.beta.mean(0, keepdim=True)
        return torch.exp(lg[dataset_id]).unsqueeze(-1), bt[dataset_id].unsqueeze(-1)

    def forward(self, x, dataset_id):
        g, b = self.params(dataset_id)
        return x * g + b

    def inverse(self, x, dataset_id):
        g, b = self.params(dataset_id)
        return (x - b) / g


# ---------------------------------------------- encoder / decoder

class Isolated_OSLO_Encoder(nn.Module):
    """Encode each frequency bin independently -> latent (B, F, latent_dim)"""
    def __init__(self, nside=16, latent_dim_per_freq=64):
        super().__init__()
        self.npix = hp.nside2npix(nside)
        self.latent_dim = latent_dim_per_freq
        self.enc1 = OSLOResBlock(1, 16, nside)
        self.enc2 = OSLOResBlock(16, 32, nside)
        self.fc_encode = nn.Sequential(
            nn.Flatten(), nn.Linear(32 * self.npix, 512), nn.GELU(),
            nn.Linear(512, self.latent_dim))

    def forward(self, x):
        B, F, N = x.shape
        z = self.fc_encode(self.enc2(self.enc1(x.reshape(B * F, 1, N))))
        return z.reshape(B, F, self.latent_dim)


class Conditioned_OSLO_Decoder(nn.Module):
    """FFM frequency-conditioned decoder + per-dataset FiLM on the output"""
    def __init__(self, nside=16, latent_dim_per_freq=64, freq_embed_dim=32,
                 num_datasets=1, num_bins=93):
        super().__init__()
        self.npix = hp.nside2npix(nside)
        self.latent_dim = latent_dim_per_freq
        self.freq_embed_dim = freq_embed_dim
        self.ffm_B = nn.Parameter(torch.randn(freq_embed_dim // 2))
        self.fc_decode = nn.Sequential(
            nn.Linear(self.latent_dim + freq_embed_dim, 512), nn.GELU(),
            nn.Linear(512, 32 * self.npix))
        self.dec1 = OSLOResBlock(32, 16, nside)
        self.dec_final = OSLOConv(16, 1, nside)
        self.film = DatasetFiLM(num_datasets, num_bins)

    def get_freq_embedding(self, num_freqs, device):
        f = torch.arange(num_freqs, device=device).float() / (num_freqs - 1)
        proj = 2 * np.pi * f.unsqueeze(1) * self.ffm_B.unsqueeze(0)
        return torch.cat([torch.sin(proj), torch.cos(proj)], dim=1)

    def forward(self, z, dataset_id=None):
        B, F, _ = z.shape
        fe = self.get_freq_embedding(F, z.device).unsqueeze(0).expand(B, -1, -1)
        h = torch.cat([z, fe], dim=-1).reshape(B * F, -1)
        h = self.fc_decode(h).reshape(B * F, 32, self.npix)
        out = self.dec_final(self.dec1(h)).reshape(B, F, self.npix)
        if dataset_id is not None:
            out = self.film(out, dataset_id)
        return out


class FFM_OSLO_Autoencoder(nn.Module):
    """One SHARED-parameter inverse FiLM at the encoder input — zero extra parameters.

    With FiLM on the decoder output only, the colouration stays in x, the encoder takes it in, and it
    lands in z — but z is exactly what the downstream geometry->latent stage needs. Output-side FiLM
    only makes the reconstruction loss line up; it does not clean z.

    Here     x_norm = (x - beta)/gamma        <- remove colouration
             z      = encoder(x_norm)
             out    = gamma * decoder(z) + beta   <- back to the original space, loss still on raw

    The SAME FiLM parameters are used. Together with centring this also kills the trivial degenerate
    direction "scale gamma up, scale the network down". Set tie_encoder_film=False to disable.
    """
    def __init__(self, nside=16, latent_dim_per_freq=64, freq_embed_dim=32,
                 num_datasets=1, num_bins=93, tie_encoder_film=True):
        super().__init__()
        self.encoder = Isolated_OSLO_Encoder(nside, latent_dim_per_freq)
        self.decoder = Conditioned_OSLO_Decoder(nside, latent_dim_per_freq,
                                                freq_embed_dim, num_datasets, num_bins)
        self.tie_encoder_film = tie_encoder_film

    def forward(self, x, dataset_id=None):
        if dataset_id is not None and self.tie_encoder_film:
            x = self.decoder.film.inverse(x, dataset_id)
        z = self.encoder(x)
        return self.decoder(z, dataset_id), z


print('model defined (FFM + FiLM)')

In [ ]:
# ==========================================================================
# Cell 3 — latent 缓存 · 受试者 · 目标 Y（每只耳一行）
# ==========================================================================
L = np.load(PATHS['latents'])
SID = list(L['sid']); AE_PART = dict(zip(SID, L['ae_part'])); IDX = {s: i for i, s in enumerate(SID)}
DEV_ALL = [str(s) for s in L['eval_sids']]; LOCKED = {str(s) for s in L['locked_sids']}
assert len(DEV_ALL) == 164 and len(LOCKED) == 36 and not (set(DEV_ALL) & LOCKED)
assert all(AE_PART[s] in ('train', 'val') for s in DEV_ALL) and all(AE_PART[s] == 'test' for s in LOCKED)

with open(os.path.join(PATHS['samples'], 'manifest.csv'), newline='') as f:
    MAN = {f"SON_{r['subject']}": r for r in csv.DictReader(f)}
mesh_hash = hashlib.md5(','.join(sorted(MAN)).encode()).hexdigest()[:12]
assert mesh_hash == str(L['mesh_hash']), f'manifest 与 FFM_3_1 用的不是同一批网格: {mesh_hash} vs {L["mesh_hash"]}'

DEV = DEV_ALL[:CFG['SMOKE_N']] if CFG['SMOKE_N'] else DEV_ALL
NS = len(DEV)
# 行 = (受试者, 耳)；行 2k 是第 k 人左耳，2k+1 是右耳（已镜像）
SUBJ_OF_ROW = np.repeat(np.arange(NS), 2); SIDE_OF_ROW = np.tile([0, 1], NS)
Y = np.stack([L['Z_ear'][IDX[s], side].reshape(-1) for s in DEV for side in (0, 1)]).astype(np.float64)
N_BINS = L['Z_ear'].shape[2]
REF_ROWS = {r['sid']: r for r in json.loads(str(L['ref_rows']))}
print(f'开发集 {NS} 人 / {2*NS} 只耳 | Y {Y.shape} | 锁定 {len(LOCKED)} 人（本 notebook 不读）'
      + (f' | ★ SMOKE_N={NS}' if CFG['SMOKE_N'] else ''))

In [ ]:
# ==========================================================================
# Cell 4 — 特征：9 个笨几何量 + 耳廓 RFF + 整头 RFF
#
# 坐标：mm，原点双耳道中点，+x 前 +y 左 +z 上。右耳那一行把整个几何镜像（y -> -y），
# 让「同侧」永远在 +y —— 与 HRTF 的右耳镜像一致。耳廓 RFF 用相对耳道的坐标。
# ==========================================================================
zmins = np.array([float(MAN[s]['z_min']) for s in DEV])
assert CFG['Z_CUT'] > zmins.max(), f"Z_CUT={CFG['Z_CUT']} 低于某人的截断高度 {zmins.max():.0f}，公共区域不成立"

FLIP = np.array([1.0, -1.0, 1.0])
G0 = np.random.default_rng(12345).standard_normal((CFG['M'], 3))   # 所有 σ 共用同一组方向，σ 只缩放

# RFF 在 GPU 上算：每组点只做一次 P @ G0ᵀ，各 σ 只是乘一个标量
import torch
_dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
_G0T = torch.as_tensor(G0.T, dtype=torch.float32, device=_dev)
@torch.no_grad()
def rff_all(P):
    """P: (n, 3) mm -> {σ: (2M,) 均值嵌入}"""
    base = 2 * np.pi * (torch.as_tensor(P, dtype=torch.float32, device=_dev) @ _G0T)
    return {sg: torch.cat([torch.cos(sg * base).mean(0), torch.sin(sg * base).mean(0)]).double().cpu().numpy()
            for sg in CFG['SIGMAS']}
print(f'RFF 在 {_dev} 上计算')

def pct(a, q): return float(np.percentile(a, q))

SIMPLE_NAMES = ['canal_x', 'canal_y', 'canal_z', 'pinna_protrusion', 'pinna_height', 'pinna_length',
                'crop_area', 'head_depth', 'head_height']
EAR_P, HEAD_P, SIMPLE = [], [], []
for s in DEV:
    d = np.load(os.path.join(PATHS['samples'], f'{s[4:]}.npz'))
    head = d['head_pts'][:CFG['N_HEAD_PTS']].astype(np.float64)
    head = head[head[:, 2] > CFG['Z_CUT']]
    for side in (0, 1):
        if side == 0:
            pts, canal, area = d['earL_pts'], d['ear_left_mm'].astype(np.float64), float(MAN[s]['crop_area_L_mm2'])
            hd = head
        else:                       # earR_pts 已镜像；耳道位置和整头要在这里镜像
            pts, canal, area = d['earR_pts'], d['ear_right_mm'].astype(np.float64) * FLIP, float(MAN[s]['crop_area_R_mm2'])
            hd = head * FLIP
        assert canal[1] > 0, f'{s} side {side}: 镜像后耳道应在 +y'
        q = pts[:CFG['N_EAR_PTS']].astype(np.float64) - canal
        out = q[q[:, 1] > CFG['PINNA_DY']]
        assert len(out) > 0.05 * len(q), f'{s} side {side}: 伸出部分只占 {len(out)}/{len(q)} 个点'
        SIMPLE.append([canal[0], canal[1], canal[2], pct(q[:, 1], 95),
                       pct(out[:, 2], 95) - pct(out[:, 2], 5), pct(out[:, 0], 95) - pct(out[:, 0], 5),
                       area, pct(hd[:, 0], 99.5) - pct(hd[:, 0], 0.5), pct(hd[:, 2], 99.9)])
        EAR_P.append(q); HEAD_P.append(hd)
SIMPLE = np.array(SIMPLE)

t0 = time.time()
EAR_F = [rff_all(q) for q in EAR_P]; HEAD_F = [rff_all(h) for h in HEAD_P]
BLOCKS = {'simple': SIMPLE}
for sg in CFG['SIGMAS']:
    BLOCKS[f'ear_s{sg}'] = np.stack([f[sg] for f in EAR_F])
    BLOCKS[f'head_s{sg}'] = np.stack([f[sg] for f in HEAD_F])
print(f'RFF 特征 {time.time()-t0:.0f} s')

print('笨特征（开发集，按耳）:')
for n, col in zip(SIMPLE_NAMES, SIMPLE.T):
    print(f'   {n:18s} median {np.median(col):8.1f}   IQR [{pct(col,25):7.1f}, {pct(col,75):7.1f}] mm{"²" if n=="crop_area" else ""}')

# ---- σ 的稳健性：耳道定位误差 1.5 mm 会让特征变多少（相对人与人之间的差别）----
print('\n耳廓 RFF 对 1.5 mm 平移的敏感度（越小越稳；>0.5 说明这个 σ 主要在编码定位误差）')
shift = RNG.standard_normal((len(EAR_P), 3)); shift *= 1.5 / np.linalg.norm(shift, axis=1, keepdims=True)
left = SIDE_OF_ROW == 0
SHIFTED = [rff_all(q + sh) for q, sh in zip(EAR_P, shift)]
for sg in CFG['SIGMAS']:
    F0 = BLOCKS[f'ear_s{sg}']; F1 = np.stack([f[sg] for f in SHIFTED])
    dd = np.linalg.norm(F1 - F0, axis=1)
    Fl = F0[left]; between = np.median(np.linalg.norm(Fl[:, None] - Fl[None], axis=-1)[np.triu_indices(left.sum(), 1)])
    print(f'   σ={sg:<6} (1/σ={1/sg:5.0f} mm)  平移造成的变化 / 人间中位距离 = {np.median(dd)/between:.3f}')
del EAR_P, HEAD_P, EAR_F, HEAD_F, SHIFTED

In [ ]:
# ==========================================================================
# Cell 5 — 配置 → Gram 矩阵
#
# 每一列在开发集上 z-score（只用特征，不碰目标）。每个块除以它的列数，
# 这样多尺度拼接时每个 σ 的权重相同。置换检验只需要置换 Gram 的行列，不用重算特征。
# ==========================================================================
def zscore(X):
    sd = X.std(0)
    keep = sd > 1e-12 * max(sd.max(), 1e-300)
    return (X[:, keep] - X[:, keep].mean(0)) / sd[keep]

GB = {}
for k, X in BLOCKS.items():
    Xz = zscore(X)
    GB[k] = Xz @ Xz.T / Xz.shape[1]

S = CFG['SIGMAS']
CONFIGS = {'simple': ['simple']}
CONFIGS.update({f'ear σ={s}': [f'ear_s{s}'] for s in S})
CONFIGS['ear multi-σ'] = [f'ear_s{s}' for s in S]
CONFIGS.update({f'head σ={s}': [f'head_s{s}'] for s in S})
CONFIGS['head multi-σ'] = [f'head_s{s}' for s in S]
CONFIGS['ear+head multi-σ'] = CONFIGS['ear multi-σ'] + CONFIGS['head multi-σ']
CONFIGS['simple+ear multi-σ'] = ['simple'] + CONFIGS['ear multi-σ']
GRAM = {name: sum(GB[b] for b in blocks) for name, blocks in CONFIGS.items()}
print(f'{len(CONFIGS)} 个配置：', ', '.join(CONFIGS))

In [ ]:
# ==========================================================================
# Cell 6 — 交叉验证引擎（核 ridge，按人分组，内层 CV 选 α，目标做折内 PCA）
#
# 折内 PCA 对 ridge 无损：预测落在训练目标的行空间里，验证误差中行空间以外的部分
# 与 α 无关，单独作为常数加回。
# ==========================================================================
def subject_folds(subjects, k, rng):
    p = rng.permutation(subjects)
    return [np.sort(a) for a in np.array_split(p, min(k, len(p)))]

ROWS_OF = lambda subs: np.flatnonzero(np.isin(SUBJ_OF_ROW, subs))

def prep_split(tr, va):
    my = Y[tr].mean(0); Yc = Y[tr] - my
    U, s, Vt = np.linalg.svd(Yc, full_matrices=False)
    k = int((s > s[0] * 1e-9).sum()); V = Vt[:k].T
    Yv = Y[va] - my; Yv_pc = Yv @ V
    return dict(tr=tr, va=va, my=my, V=V, Ytr_pc=U[:, :k] * s[:k], Yv_pc=Yv_pc,
                resid=float((Yv ** 2).sum() - (Yv_pc ** 2).sum()), sst=float((Yv ** 2).sum()))

def build_plan(subjects, rng):
    plan = []
    for te_s in subject_folds(subjects, CFG['N_FOLDS'], rng):
        tr_s = np.setdiff1d(subjects, te_s)
        outer = prep_split(ROWS_OF(tr_s), ROWS_OF(te_s))
        inner = [prep_split(ROWS_OF(np.setdiff1d(tr_s, v)), ROWS_OF(v))
                 for v in subject_folds(tr_s, CFG['N_INNER'], rng)]
        plan.append((outer, inner))
    return plan

def centred(G, tr, va):
    Gtt, Gvt = G[np.ix_(tr, tr)], G[np.ix_(va, tr)]
    cm, tot = Gtt.mean(0), Gtt.mean()
    return Gtt - cm[None] - cm[:, None] + tot, Gvt - cm[None] - Gvt.mean(1)[:, None] + tot

def ridge_path(G, sp, alphas_rel):
    """所有 α 下验证集在 PC 坐标的预测；返回 (α 绝对值, [预测])"""
    Ktt, Kvt = centred(G, sp['tr'], sp['va'])
    lam, Q = np.linalg.eigh(Ktt); lam = np.clip(lam, 0, None)
    Bq = Q.T @ sp['Ytr_pc']; KvQ = Kvt @ Q
    scale = np.trace(Ktt) / len(sp['tr'])
    alphas = np.asarray(alphas_rel) * scale
    return alphas, [(KvQ / (lam + a)) @ Bq for a in alphas]

def sse(sp, pred_pc):
    return float(((sp['Yv_pc'] - pred_pc) ** 2).sum()) + sp['resid']

def run_cv(G, plan, want_pred=False):
    A = CFG['ALPHA_REL']; SSE = SST = 0.0; chosen = []; pred = np.zeros_like(Y) if want_pred else None
    for outer, inner in plan:
        tot = np.zeros(len(A))
        for sp in inner:
            _, preds = ridge_path(G, sp, A)
            tot += [sse(sp, p) for p in preds]
        j = int(np.argmin(tot)); chosen.append(j)
        _, preds = ridge_path(G, outer, [A[j]])
        SSE += sse(outer, preds[0]); SST += outer['sst']
        if want_pred:
            pred[outer['va']] = outer['my'] + preds[0] @ outer['V'].T
    return 1 - SSE / SST, chosen, pred

PLAN = build_plan(np.arange(NS), np.random.default_rng(CFG['SEED']))
# 断言：外层折按人不相交且覆盖全部；同一人的两只耳在同一折
_va = np.concatenate([o['va'] for o, _ in PLAN])
assert np.array_equal(np.sort(_va), np.arange(2 * NS)), '外层折没有恰好覆盖每只耳一次'
for o, inn in PLAN:
    assert not set(SUBJ_OF_ROW[o['tr']]) & set(SUBJ_OF_ROW[o['va']]), '同一人跨了训练/验证'
    for sp in inn:
        assert set(sp['tr']) | set(sp['va']) == set(o['tr']), '内层折没有恰好覆盖外层训练集'
print(f"外层 {len(PLAN)} 折 / 内层 {CFG['N_INNER']} 折，按人分组 ✓")

# 自检：特征与目标无关时 R² 应在 0 附近或为负
_Xr = np.random.default_rng(1).standard_normal((2 * NS, 300)); _Gr = _Xr @ _Xr.T / 300
print(f'自检：纯随机特征 CV-R² = {run_cv(_Gr, PLAN)[0]:+.4f}（应 ≤ ~0）')

In [ ]:
# ==========================================================================
# Cell 7 — 所有配置的 CV-R²(z)（内部信号）
# ==========================================================================
RES = {}
t0 = time.time()
for name, G in GRAM.items():
    r2, chosen, pred = run_cv(G, PLAN, want_pred=True)
    edge = sum(c in (0, len(CFG['ALPHA_REL']) - 1) for c in chosen)
    RES[name] = dict(r2=r2, alpha_idx=chosen, pred=pred)
    print(f'{name:22s} CV-R²(z) {r2:+.4f}   α 中位 {CFG["ALPHA_REL"][int(np.median(chosen))]:.2g}·tr/n'
          + (f'   ★ {edge}/{len(chosen)} 折选在 α 网格边上' if edge else ''))
BEST = max(RES, key=lambda k: RES[k]['r2'])
print(f'\n最好配置：{BEST}  CV-R²(z) = {RES[BEST]["r2"]:+.4f}   ({time.time()-t0:.0f} s)')
print('★ 这是在 15 个配置里挑出来的最大值 —— 显著性看 Cell 8 的「最大值零分布」，不是看 0。')

In [ ]:
# ==========================================================================
# Cell 8 — 置换检验：打乱「网格 ↔ 人」，重跑全部 15 个配置取最大
# 一个人的两只耳一起换（左耳特征仍配左耳目标）。
# ==========================================================================
def permuted_rows(p):
    return np.array([2 * p[k] + side for k in range(NS) for side in (0, 1)])

NULL_MAX = []
t0 = time.time()
prng = np.random.default_rng(CFG['SEED'] + 1)
for b in range(CFG['N_PERM']):
    r = permuted_rows(prng.permutation(NS))
    NULL_MAX.append(max(run_cv(G[np.ix_(r, r)], PLAN)[0] for G in GRAM.values()))
    if b == 0: print(f'每次置换约 {time.time()-t0:.1f} s')
NULL_MAX = np.array(NULL_MAX)
OBS = RES[BEST]['r2']
P_MAX = (1 + (NULL_MAX >= OBS).sum()) / (1 + len(NULL_MAX))
print(f'最大值零分布：中位 {np.median(NULL_MAX):+.4f}，95% 分位 {np.percentile(NULL_MAX, 95):+.4f}')
print(f'观测 {OBS:+.4f}  ->  置换 p = {P_MAX:.3f}  (B={len(NULL_MAX)})')

In [ ]:
# ==========================================================================
# Cell 9 — 端到端 dB · 收回比例 · 认人
#
# 基线 ④ 每折单独算：AE-train ∪ AE-val 的 SONICOM 受试者里，去掉本折留出的人，取耳均值。
# AE-test 的 60 人（含锁定的 36 人）一律不进均值。
# ==========================================================================
E2E = {}
if CFG['DECODE']:
    import torch, healpy as hp
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    ck = torch.load(PATHS['ckpt'], map_location=device, weights_only=False)
    model = FFM_OSLO_Autoencoder(nside=16, latent_dim_per_freq=64, freq_embed_dim=32,
                                 num_datasets=4, num_bins=N_BINS, tie_encoder_film=True).to(device)
    model.load_state_dict(ck['model_state_dict']); model.eval(); del ck

    def find_dir(root, target):
        c = []
        for dp, _, fns in os.walk(root):
            if '__MACOSX' in dp: continue
            if os.path.basename(dp) == target: c.append((sum(f.endswith('.npy') for f in fns), dp))
        c.sort(reverse=True); return c[0][1] if c else None
    if PATHS['sonicom'].endswith('.zip'):
        SON_OUT = '/content/dataset/sonicom'; os.makedirs(SON_OUT, exist_ok=True)
        if find_dir(SON_OUT, 'npy_left') is None:
            with zipfile.ZipFile(PATHS['sonicom']) as z: z.extractall(SON_OUT)
    else:
        SON_OUT = PATHS['sonicom']
    DIRS = (find_dir(SON_OUT, 'npy_left'), find_dir(SON_OUT, 'npy_right'))
    _th, _ph = hp.pix2ang(16, np.arange(3072)); MIRROR = hp.ang2pix(16, _th, (-_ph) % (2 * np.pi))
    def load_ear(sid, side):
        x = np.load(os.path.join(DIRS[side], f"{sid}_{('left', 'right')[side]}_full.npy")).astype(np.float32)[:N_BINS]
        return x[:, MIRROR] if side == 1 else x

    POOL = [s for s in SID if AE_PART[s] in ('train', 'val')]
    assert not set(POOL) & LOCKED
    X = np.zeros((2 * NS, N_BINS, 3072), np.float32); SUM = np.zeros((N_BINS, 3072)); dev_pos = {s: k for k, s in enumerate(DEV)}
    for s in POOL:
        for side in (0, 1):
            x = load_ear(s, side); SUM += x
            if s in dev_pos: X[2 * dev_pos[s] + side] = x
    N_POOL_EARS = 2 * len(POOL)
    FREQS = np.arange(N_BINS) * 44100 / 256
    NOTCH = (FREQS >= CFG['NOTCH_HZ'][0]) & (FREQS <= CFG['NOTCH_HZ'][1])
    # SONICOM 实测只到仰角 -45°；以下的 HEALPix 像素是 STPS 外推的
    MEAS = (90 - np.degrees(_th)) >= -45
    def lsd(e2):
        """文献口径 LSD：每个方向在频率上取 RMS，再对方向取算术平均。返回 (全球面, 仰角≥-45°)，每只耳一个值"""
        d = np.sqrt(e2.mean(1))
        return d.mean(1), d[:, MEAS].mean(1)

    # 每只耳的基线：去掉它所在折的留出受试者后的均值
    BASE_MSE = np.zeros((2 * NS, 2)); BASE_BIN = np.zeros((2 * NS, N_BINS)); BASE_LSD = np.zeros((2 * NS, 2))
    for outer, _ in PLAN:
        va = outer['va']; mu = (SUM - X[va].sum(0)) / (N_POOL_EARS - len(va))
        e2 = (X[va] - mu) ** 2
        BASE_MSE[va, 0] = e2.mean((1, 2)); BASE_MSE[va, 1] = e2[:, NOTCH].mean((1, 2)); BASE_BIN[va] = e2.mean(2)
        BASE_LSD[va] = np.stack(lsd(e2), 1)

    @torch.no_grad()
    def decode_rows(Zrows):
        out = np.zeros((len(Zrows), N_BINS, 3072), np.float32)
        for b in range(0, len(Zrows), 16):
            z = torch.from_numpy(Zrows[b:b + 16].reshape(-1, N_BINS, 64).astype(np.float32)).to(device)
            out[b:b + 16] = model.decoder(z, torch.full((len(z),), CFG['DID'], device=device)).cpu().numpy()
        return out

    def pooled(ms): return float(np.sqrt(ms.mean()))
    def subj_pooled(per_ear):            # 每人两只耳平均，再 pooled
        return pooled(per_ear.reshape(NS, 2).mean(1))

    @torch.no_grad()
    def retrieval(P):
        """每侧：预测 HRTF 与 164 个真实同侧 HRTF 的 RMSE；本人排第几"""
        ranks = []
        for side in (0, 1):
            r = np.flatnonzero(SIDE_OF_ROW == side)
            a = torch.from_numpy(P[r].reshape(len(r), -1)).to(device)
            b = torch.from_numpy(X[r].reshape(len(r), -1)).to(device)
            d2 = (a * a).sum(1)[:, None] + (b * b).sum(1)[None] - 2 * a @ b.T
            ranks.append((d2 < d2.diagonal()[:, None]).sum(1).cpu().numpy() + 1)
        return np.concatenate(ranks)

    REC = np.array([REF_ROWS[s]['recon'] for s in DEV]); REC_N = np.array([REF_ROWS[s]['recon_notch'] for s in DEV])
    B_ALL, B_NOTCH = subj_pooled(BASE_MSE[:, 0]), subj_pooled(BASE_MSE[:, 1])
    R_ALL, R_NOTCH = pooled(REC), pooled(REC_N)

    # 认人指标的自检：用 AE 自己的重建去认，应当几乎全对
    _rec = decode_rows(Y); _rk = retrieval(_rec); REC_LSD = np.stack(lsd((_rec - X) ** 2), 1); del _rec
    print(f'认人自检（AE 重建）：rank-1 {np.mean(_rk == 1):.1%} —— 若明显低于 ~90%，这个指标本身有问题')

    names = list(CONFIGS) if CFG['E2E_CONFIGS'] == 'all' else list(CFG['E2E_CONFIGS'])
    if BEST not in names: names.append(BEST)
    for name in names:
        P = decode_rows(RES[name]['pred'])
        e2 = (P - X) ** 2
        m_all, m_n = subj_pooled(e2.mean((1, 2))), subj_pooled(e2[:, NOTCH].mean((1, 2)))
        rk = retrieval(P)
        l_all, l_meas = lsd(e2)
        E2E[name] = dict(db=m_all, db_notch=m_n, lsd=float(l_all.mean()), lsd_meas=float(l_meas.mean()),
                         recovered=(B_ALL - m_all) / (B_ALL - R_ALL),
                         recovered_notch=(B_NOTCH - m_n) / (B_NOTCH - R_NOTCH),
                         rank1=float(np.mean(rk == 1)), top5=float(np.mean(rk <= 5)),
                         median_rank=float(np.median(rk)))
        if name == BEST: BEST_BIN = e2.mean(2)
        del P, e2

    print(f"\n{'':22s}{'CV-R²(z)':>9s}{'端到端':>8s}{'6–12k':>8s}{'收回':>7s}{'6–12k收回':>10s}{'认人':>7s}{'前5':>6s}{'中位名次':>8s}{'LSD':>7s}{'LSD≥-45°':>10s}")
    print(f"{'④ 均值 HRTF 基线':22s}{'':>9s}{B_ALL:8.3f}{B_NOTCH:8.3f}{'0%':>7s}{'0%':>10s}{'':>7s}{'':>6s}{'':>8s}"
          f"{BASE_LSD[:, 0].mean():7.3f}{BASE_LSD[:, 1].mean():10.3f}")
    for name in names:
        e = E2E[name]
        print(f"{name:22s}{RES[name]['r2']:+9.4f}{e['db']:8.3f}{e['db_notch']:8.3f}{e['recovered']:7.1%}"
              f"{e['recovered_notch']:10.1%}{e['rank1']:7.1%}{e['top5']:6.1%}{e['median_rank']:8.0f}{e['lsd']:7.3f}{e['lsd_meas']:10.3f}")
    print(f"{'① AE 重建（下限）':22s}{'':>9s}{R_ALL:8.3f}{R_NOTCH:8.3f}{'100%':>7s}{'100%':>10s}{'':>7s}{'':>6s}{'':>8s}"
          f"{REC_LSD[:, 0].mean():7.3f}{REC_LSD[:, 1].mean():10.3f}")
    print(f'\n认人的随机水平：rank-1 {1/NS:.1%}，前 5 {5/NS:.1%}，中位名次 {(NS+1)/2:.0f}')
    print('LSD = 每个方向在 DC–15.85 kHz 的 93 个 bin 上取 RMS，再对方向、对耳取算术平均（文献口径，'
          '总是小于 pooled RMSE）。≥-45° 只算实测覆盖到的方向。与论文比较时仍要核对频带、方向网格和数据库。')
else:
    print('DECODE=False：跳过端到端（只有 CV-R²）')

In [ ]:
# ==========================================================================
# Cell 10 — 学习曲线（最好配置，CV-R²）：是「几何没信息」还是「人不够」
# ==========================================================================
LC = {}
G = GRAM[BEST]
for n in CFG['LC_SIZES']:
    if n > NS: continue
    reps = 1 if n == NS else CFG['LC_REPEATS']
    vals = []
    for rep in range(reps):
        rr = np.random.default_rng(1000 * n + rep)
        subs = np.sort(rr.choice(NS, n, replace=False))
        vals.append(run_cv(G, build_plan(subs, rr))[0])
    LC[n] = vals
    print(f'N={n:3d} 人：CV-R²(z) {np.mean(vals):+.4f} ± {np.std(vals):.4f}  ({reps} 次)')

In [ ]:
# ==========================================================================
# Cell 11 — 信号在哪个频段（最好配置）
# ==========================================================================
import matplotlib.pyplot as plt
FREQS = np.arange(N_BINS) * 44100 / 256
pred = RES[BEST]['pred'].reshape(-1, N_BINS, 64); Yb = Y.reshape(-1, N_BINS, 64)
# 每个 bin 的 R²：分母用每折训练均值
SSEb = np.zeros(N_BINS); SSTb = np.zeros(N_BINS)
for outer, _ in PLAN:
    va = outer['va']; my = outer['my'].reshape(N_BINS, 64)
    SSEb += ((Yb[va] - pred[va]) ** 2).sum((0, 2)); SSTb += ((Yb[va] - my) ** 2).sum((0, 2))
r2_bin = 1 - SSEb / SSTb

fig, ax = plt.subplots(1, 2 if CFG['DECODE'] else 1, figsize=(12 if CFG['DECODE'] else 6, 3.6), squeeze=False)
ax[0, 0].plot(FREQS / 1000, r2_bin, lw=1.5); ax[0, 0].axhline(0, c='k', lw=.6)
ax[0, 0].set_title(f'CV-R²(z) per frequency — {BEST}'); ax[0, 0].set_xlabel('kHz')
if CFG['DECODE']:
    gain = np.sqrt(BASE_BIN.mean(0)) - np.sqrt(BEST_BIN.mean(0))
    ax[0, 1].plot(FREQS / 1000, gain, lw=1.5); ax[0, 1].axhline(0, c='k', lw=.6)
    ax[0, 1].set_title('dB better than mean-HRTF baseline'); ax[0, 1].set_xlabel('kHz')
for a in ax.ravel():
    a.axvspan(6, 12, color='grey', alpha=.12); a.grid(alpha=.3)
plt.tight_layout(); plt.savefig(os.path.join(OUT_DIR, 'per_frequency.png'), dpi=150); plt.show()

In [ ]:
# ==========================================================================
# Cell 12 — 结论 · 导出
# ==========================================================================
stop = OBS < CFG['STOP_R2']
print(f"最好配置 {BEST}：CV-R²(z) {OBS:+.4f}，最大值置换 p = {P_MAX:.3f}")
print(f"停止规则（CV-R² < {CFG['STOP_R2']}）：{'★ 触发 —— 不加容量，写上界结论' if stop else '未触发 —— 可以考虑更强的模型'}")
if CFG['DECODE']:
    e = E2E[BEST]
    print(f"端到端 {e['db']:.3f} dB（基线 {B_ALL:.3f}，AE 重建 {R_ALL:.3f}）—— 收回 {e['recovered']:.1%}；"
          f"6–12 kHz 收回 {e['recovered_notch']:.1%}；认人 rank-1 {e['rank1']:.1%}（随机 {1/NS:.1%}）")
out = dict(
    cfg={k: (list(map(float, v)) if isinstance(v, list) and v and isinstance(v[0], float) else v) for k, v in CFG.items()},
    n_subjects=NS, smoke=bool(CFG['SMOKE_N']), best=BEST, best_r2=OBS, p_max=P_MAX,
    null_max=list(map(float, NULL_MAX)), stop_rule_triggered=bool(stop),
    cv_r2={k: v['r2'] for k, v in RES.items()},
    alpha_idx={k: v['alpha_idx'] for k, v in RES.items()},
    e2e=E2E,
    baseline=dict(all=B_ALL, notch=B_NOTCH, lsd=float(BASE_LSD[:, 0].mean()), lsd_meas=float(BASE_LSD[:, 1].mean()))
             if CFG['DECODE'] else None,
    recon=dict(all=R_ALL, notch=R_NOTCH, lsd=float(REC_LSD[:, 0].mean()), lsd_meas=float(REC_LSD[:, 1].mean()))
          if CFG['DECODE'] else None,
    learning_curve={str(k): v for k, v in LC.items()}, r2_per_bin=list(map(float, r2_bin)),
    latents_mesh_hash=str(L['mesh_hash']), latents_subj_hash=str(L['subj_hash']),
)
with open(os.path.join(OUT_DIR, 'results.json'), 'w') as f:
    json.dump(out, f, indent=1, ensure_ascii=False, default=float)
print(f'-> {OUT_DIR}/results.json')

# 交叉验证的预测和折划分，供 FFM_3_3 在完全相同的折上做逐人配对比较
FOLD_OF_SUBJECT = np.full(NS, -1)
for f, (outer, _) in enumerate(PLAN):
    FOLD_OF_SUBJECT[np.unique(SUBJ_OF_ROW[outer['va']])] = f
assert (FOLD_OF_SUBJECT >= 0).all()
np.savez_compressed(os.path.join(OUT_DIR, 'oof_ridge.npz'),
                    dev=np.array(DEV), fold_of_subject=FOLD_OF_SUBJECT, best=BEST,
                    pred_best=RES[BEST]['pred'].astype(np.float32),
                    pred_simple=RES['simple']['pred'].astype(np.float32),
                    latents_subj_hash=str(L['subj_hash']), latents_mesh_hash=str(L['mesh_hash']))
print(f'-> {OUT_DIR}/oof_ridge.npz（{BEST} 与 simple 的交叉验证预测、折划分）')

## 交付前检查

1. **Cell 6 自检**：纯随机特征的 CV-R² 必须 ≤ 约 0。为正说明管线泄漏。
2. **Cell 7**：若很多折把 α 选在网格边上，把 `ALPHA_REL` 的范围放宽再跑。
3. **Cell 8**：报告的是**最大值置换 p**，不是最好配置对 0 的比较。
4. **Cell 9 认人自检**：AE 自己的重建 rank-1 应接近 100%。否则认人指标不可信。
5. **Cell 4 平移敏感度**：大于 0.5 的 σ 主要在编码耳道定位误差，即使它分数高也要谨慎解释。
6. 选择偏差：开发集来自扫描质量合格的 200 人（FFM_3_1 Cell 5 ③，p=0.030），结论只覆盖这部分人。
7. **锁定的 36 人没有被读取**。它们只在所有选择冻结之后用一次。